# 07 Dynamically Load Adapters into vLLM

This notebook loads the local PEFT adapters into a running vLLM server.

```mermaid
sequenceDiagram
    participant N as Notebook
    participant V as vLLM
    N->>V: POST /v1/load_lora_adapter
    N->>V: GET /v1/models
    V-->>N: base plus LoRA model names
```

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "PROJECT_SPEC.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT))

from llmops_demo.settings import settings, ensure_dirs

cfg = settings()
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model: {cfg.base_model}")
print(f"Adapters: {cfg.adapters}")

## Load adapters

Expected output: one success line per adapter.

In [ ]:
from scripts.load_adapters import load_adapter

for adapter in cfg.adapters:
    load_adapter(cfg.vllm_base_url, cfg.vllm_api_key, adapter, cfg.adapter_dir / adapter)

## Verify vLLM model registration

Expected output: `finance`, `legal`, and `healthcare` appear in `/v1/models`.

In [ ]:
import requests

response = requests.get(f"{cfg.vllm_base_url}/v1/models", headers={"Authorization": f"Bearer {cfg.vllm_api_key}"}, timeout=10)
response.raise_for_status()
print(response.json())